# 12 — Silver Data Quality: Validação Consolidada

**Credit Risk Intelligence Platform** — Camada Silver

Este notebook realiza a validação centralizada da qualidade dos dados de todas as tabelas Silver, avaliando se a camada está pronta para alimentar a camada Gold e posteriormente o Machine Learning.

## Tabelas avaliadas

* `credit_risk.silver.application_train`
* `credit_risk.silver.application_test`
* `credit_risk.silver.bureau`
* `credit_risk.silver.bureau_balance`
* `credit_risk.silver.previous_application`
* `credit_risk.silver.pos_cash_balance`
* `credit_risk.silver.credit_card_balance`
* `credit_risk.silver.installments_payments`

## Dimensões avaliadas

| Dimensão | Descrição |
|----------|-----------|
| **Completeness** | NULLs e completude por coluna |
| **Uniqueness** | Duplicidades completas e por chave |
| **Validity** | Tipos, valores negativos, domínios, extremos |
| **Consistency** | Consistência interna de cada tabela |
| **Referential Integrity** | Integridade entre tabelas (chaves estrangeiras) |

## Regras

> **Nenhuma tabela Silver é modificada** durante este notebook.
> Resultados são gravados em `credit_risk.silver.data_quality` e `credit_risk.silver.data_quality_summary` (APPEND).
> Duplicidades legítimas (ex: pagamentos parciais em installments_payments) são distinguidas de violações de chave.
> Outliers são identificados mas não removidos.
> Quality Score é transparente, documentado e configurável.

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Imports e Parâmetros
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from pyspark.sql.types import (StructType, StructField, StringType, 
    IntegerType, DoubleType, TimestampType, LongType, DateType)
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Identificadores de execução
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "silver_dq_v1.0"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"silver_dq_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# Thresholds configuráveis (Quality Score)
# ----------------------------------------------------------------------------
THRESHOLDS = {
    # Completeness
    "completeness_pass": 95.0,       # >= 95% completeness → PASS
    "completeness_warning": 80.0,    # >= 80% → WARNING, < 80% → FAIL
    
    # Uniqueness
    "uniqueness_pass": 99.0,          # <= 1% duplicates → PASS
    "uniqueness_warning": 95.0,       # <= 5% duplicates → WARNING, > 5% → FAIL
    
    # Validity
    "validity_pass": 99.0,            # >= 99% valid → PASS
    "validity_warning": 95.0,        # >= 95% → WARNING, < 95% → FAIL
    
    # Consistency
    "consistency_pass": 99.0,
    "consistency_warning": 95.0,
    
    # Referential Integrity
    "ref_integrity_pass": 99.0,       # >= 99% matched → PASS
    "ref_integrity_warning": 95.0,    # >= 95% → WARNING, < 95% → FAIL
    
    # Overall Score
    "score_pass": 90.0,               # >= 90 → PASS
    "score_warning": 75.0,            # >= 75 → WARNING, < 75 → FAIL
}

# Pesos das dimensões no Quality Score
DIMENSION_WEIGHTS = {
    "completeness": 0.30,
    "uniqueness": 0.20,
    "validity": 0.20,
    "consistency": 0.15,
    "referential_integrity": 0.15,
}

# ----------------------------------------------------------------------------
# Definição das tabelas Silver e Bronze
# ----------------------------------------------------------------------------
SILVER_TABLES = [
    "application_train",
    "application_test",
    "bureau",
    "bureau_balance",
    "previous_application",
    "pos_cash_balance",
    "credit_card_balance",
    "installments_payments",
]

BRONZE_TABLES = {t: f"credit_risk.bronze.{t}" for t in SILVER_TABLES}
SILVER_TABLES_MAP = {t: f"credit_risk.silver.{t}" for t in SILVER_TABLES}

# ----------------------------------------------------------------------------
# Chaves por tabela
# ----------------------------------------------------------------------------
TABLE_KEYS = {
    "application_train": {"pk": ["SK_ID_CURR"], "note": "PK unica"},
    "application_test": {"pk": ["SK_ID_CURR"], "note": "PK unica"},
    "bureau": {"pk": ["SK_ID_CURR", "SK_ID_BUREAU"], "note": "PK composta"},
    "bureau_balance": {"pk": ["SK_ID_BUREAU", "MONTHS_BALANCE"], "note": "PK composta (pode ter repeticao)"},
    "previous_application": {"pk": ["SK_ID_PREV"], "note": "PK unica"},
    "pos_cash_balance": {"pk": ["SK_ID_PREV", "MONTHS_BALANCE"], "note": "PK composta (pode ter repeticao)"},
    "credit_card_balance": {"pk": ["SK_ID_PREV", "MONTHS_BALANCE"], "note": "PK composta (pode ter repeticao)"},
    "installments_payments": {"pk": ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"], "note": "Nao unica — pagamentos parciais legitimos"},
}

# ----------------------------------------------------------------------------
# Relacionamentos referenciais
# ----------------------------------------------------------------------------
REFERENTIAL_RELATIONSHIPS = [
    {"source": "application_train", "source_key": "SK_ID_CURR", 
     "target": "bureau", "target_key": "SK_ID_CURR"},
    {"source": "application_train", "source_key": "SK_ID_CURR", 
     "target": "previous_application", "target_key": "SK_ID_CURR"},
    {"source": "application_train", "source_key": "SK_ID_CURR", 
     "target": "pos_cash_balance", "target_key": "SK_ID_CURR"},
    {"source": "application_train", "source_key": "SK_ID_CURR", 
     "target": "credit_card_balance", "target_key": "SK_ID_CURR"},
    {"source": "application_train", "source_key": "SK_ID_CURR", 
     "target": "installments_payments", "target_key": "SK_ID_CURR"},
    {"source": "application_test", "source_key": "SK_ID_CURR", 
     "target": "bureau", "target_key": "SK_ID_CURR"},
    {"source": "application_test", "source_key": "SK_ID_CURR", 
     "target": "previous_application", "target_key": "SK_ID_CURR"},
    {"source": "bureau", "source_key": "SK_ID_BUREAU", 
     "target": "bureau_balance", "target_key": "SK_ID_BUREAU"},
    {"source": "previous_application", "source_key": "SK_ID_PREV", 
     "target": "pos_cash_balance", "target_key": "SK_ID_PREV"},
    {"source": "previous_application", "source_key": "SK_ID_PREV", 
     "target": "credit_card_balance", "target_key": "SK_ID_PREV"},
    {"source": "previous_application", "source_key": "SK_ID_PREV", 
     "target": "installments_payments", "target_key": "SK_ID_PREV"},
]

# ----------------------------------------------------------------------------
# Acumulador de resultados DQ
# ----------------------------------------------------------------------------
DQ_RESULTS = []
TABLE_DIMENSION_SCORES = {}  # {table_name: {dimension: score}}
TABLE_INFO_CACHE = {}  # Cache de informações das tabelas

# ----------------------------------------------------------------------------
# Tabelas de destino
# ----------------------------------------------------------------------------
DQ_DETAIL_TABLE = "credit_risk.silver.data_quality"
DQ_SUMMARY_TABLE = "credit_risk.silver.data_quality_summary"
AUDIT_TABLE = "credit_risk.silver.audit_transformation"

# Criar schema se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.silver")

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"📊 Tabelas a avaliar: {len(SILVER_TABLES)}")
print(f"🔗 Relacionamentos: {len(REFERENTIAL_RELATIONSHIPS)}")
print("✅ Configuração inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 2 — Inspeção das Tabelas Silver
# ============================================================================
# Verifica existencia, schema, row_count, col_count, NULLs e duplicidades.

sep = "─" * 70
print("=" * 70)
print("INSPEÇÃO DAS TABELAS SILVER")
print("=" * 70)

def add_dq(table, column, dimension, metric, value, threshold, status, rule, desc):
    """Adiciona um resultado DQ ao acumulador."""
    DQ_RESULTS.append({
        "execution_timestamp": EXECUTION_TIMESTAMP,
        "execution_id": EXECUTION_ID,
        "table_name": table,
        "column_name": column,
        "quality_dimension": dimension,
        "metric_name": metric,
        "metric_value": float(value) if value is not None else None,
        "threshold": float(threshold) if threshold is not None else None,
        "status": status,
        "rule_name": rule,
        "rule_description": desc,
    })

for tname in SILVER_TABLES:
    full_name = SILVER_TABLES_MAP[tname]
    print(f"\n{'─' * 60}")
    print(f"📊 {full_name}")
    print(f"{'─' * 60}")
    
    try:
        df = spark.table(full_name)
        rc = df.count()
        cc = len(df.columns)
        cols = [(f.name, f.dataType.simpleString(), f.nullable) for f in df.schema.fields]
        
        TABLE_INFO_CACHE[tname] = {
            "df": df, "row_count": rc, "col_count": cc,
            "columns": cols, "exists": True
        }
        
        # Total de NULLs
        null_exprs = [F.sum(F.when(F.col(c[0]).isNull(), 1).otherwise(0)) for c in cols]
        null_result = df.agg(*null_exprs).collect()[0]
        total_nulls = sum([null_result[i] for i in range(len(cols))])
        
        # Duplicidade completa
        full_dups = rc - df.dropDuplicates().count()
        
        print(f"   Registros: {rc:,}")
        print(f"   Colunas: {cc}")
        print(f"   NULLs totais: {total_nulls:,}")
        print(f"   Duplicidades completas: {full_dups}")
        print(f"   Schema (primeiras 5 colunas):")
        for cn, ct, nb in cols[:5]:
            print(f"      {cn:<35} {ct:<12} nullable={nb}")
        if cc > 5:
            print(f"      ... +{cc-5} colunas")
        
        # Registrar metricas de inspecao
        add_dq(tname, None, "inspection", "row_count", rc, 0, "PASS", "row_count_check", f"Total de registros: {rc}")
        add_dq(tname, None, "inspection", "column_count", cc, 0, "PASS", "column_count_check", f"Total de colunas: {cc}")
        add_dq(tname, None, "inspection", "total_nulls", total_nulls, 0, "PASS", "null_count_check", f"NULLs totais: {total_nulls}")
        add_dq(tname, None, "inspection", "full_duplicates", full_dups, 0, "PASS" if full_dups == 0 else "WARNING", "full_dup_check", f"Duplicidades completas: {full_dups}")
        
    except Exception as e:
        TABLE_INFO_CACHE[tname] = {"exists": False, "error": str(e)}
        print(f"   ❌ ERRO: {e}")
        add_dq(tname, None, "inspection", "table_exists", 0, 1, "FAIL", "table_exists_check", f"Tabela nao existe: {e}")

print(f"\n✅ Inspeção concluída para {len(SILVER_TABLES)} tabelas!")

In [0]:
# ============================================================================
# CÉLULA 3 — Completeness (NULLs por coluna)
# ============================================================================
# Calcula completude por coluna para cada tabela Silver.

sep = "─" * 70
print("=" * 70)
print("COMPLETENESS — NULLs POR COLUNA")
print("=" * 70)

for tname in SILVER_TABLES:
    info = TABLE_INFO_CACHE.get(tname, {})
    if not info.get("exists"):
        continue
    
    df = info["df"]
    rc = info["row_count"]
    cols = info["columns"]
    
    print(f"\n{'─' * 60}")
    print(f"📊 {tname} ({rc:,} registros, {len(cols)} colunas)")
    print(f"{'─' * 60}")
    
    # Calcular NULLs por coluna em uma única passagem
    null_exprs = [F.sum(F.when(F.col(c[0]).isNull(), 1).otherwise(0)).alias(c[0]) for c in cols]
    null_row = df.agg(*null_exprs).collect()[0]
    
    col_scores = []
    for cn, ct, nb in cols:
        null_cnt = null_row[cn] if null_row[cn] else 0
        null_pct = (null_cnt / rc * 100) if rc > 0 else 0
        completeness_pct = 100 - null_pct
        non_null = rc - null_cnt
        
        # Classificação
        if completeness_pct >= THRESHOLDS["completeness_pass"]:
            status = "PASS"
        elif completeness_pct >= THRESHOLDS["completeness_warning"]:
            status = "WARNING"
        else:
            status = "FAIL"
        
        if null_cnt > 0:
            print(f"   {cn:<40} null={null_cnt:>10,} ({null_pct:.2f}%) → {status}")
        
        add_dq(tname, cn, "completeness", "null_count", null_cnt, 0, status, 
               "null_count_check", f"NULLs: {null_cnt} ({null_pct:.2f}%)")
        add_dq(tname, cn, "completeness", "null_percentage", null_pct, 
               100 - THRESHOLDS["completeness_pass"], status,
               "null_pct_check", f"% NULL: {null_pct:.2f}% (threshold: < {100 - THRESHOLDS['completeness_pass']}%)")
        add_dq(tname, cn, "completeness", "completeness_percentage", completeness_pct,
               THRESHOLDS["completeness_pass"], status,
               "completeness_check", f"Completude: {completeness_pct:.2f}%")
        
        col_scores.append(completeness_pct)
    
    # Score de completeness da tabela
    table_completeness = sum(col_scores) / len(col_scores) if col_scores else 100
    TABLE_DIMENSION_SCORES.setdefault(tname, {})["completeness"] = table_completeness
    
    if table_completeness >= THRESHOLDS["completeness_pass"]:
        status = "PASS"
    elif table_completeness >= THRESHOLDS["completeness_warning"]:
        status = "WARNING"
    else:
        status = "FAIL"
    
    print(f"   → Score completeness: {table_completeness:.2f}% [{status}]")
    add_dq(tname, None, "completeness", "table_completeness_score", table_completeness,
           THRESHOLDS["completeness_pass"], status,
           "table_completeness_score", f"Score médio de completude: {table_completeness:.2f}%")

print(f"\n✅ Completeness avaliado para {len(SILVER_TABLES)} tabelas!")

In [0]:
# ============================================================================
# CÉLULA 4 — Uniqueness (Duplicidades)
# ============================================================================
# Avalia duplicidade completa e por chave. Distingue repeticao legítima de violacao.

sep = "─" * 70
print("=" * 70)
print("UNIQUENESS — DUPLICIDADES")
print("=" * 70)

for tname in SILVER_TABLES:
    info = TABLE_INFO_CACHE.get(tname, {})
    if not info.get("exists"):
        continue
    
    df = info["df"]
    rc = info["row_count"]
    key_info = TABLE_KEYS.get(tname, {})
    pk_cols = key_info.get("pk", [])
    pk_note = key_info.get("note", "")
    
    print(f"\n{'─' * 60}")
    print(f"📊 {tname} — Chave: {pk_cols} ({pk_note})")
    print(f"{'─' * 60}")
    
    # Duplicidade completa
    full_dups = rc - df.dropDuplicates().count()
    full_dup_pct = (full_dups / rc * 100) if rc > 0 else 0
    
    if full_dups == 0:
        status_full = "PASS"
    elif full_dup_pct < (100 - THRESHOLDS["uniqueness_pass"]):
        status_full = "PASS"
    elif full_dup_pct < (100 - THRESHOLDS["uniqueness_warning"]):
        status_full = "WARNING"
    else:
        status_full = "FAIL"
    
    print(f"   Duplicidade completa: {full_dups:,} ({full_dup_pct:.4f}%) → {status_full}")
    add_dq(tname, None, "uniqueness", "full_duplicates", full_dups, 0, status_full,
           "full_dup_check", f"Duplicidades completas: {full_dups}")
    add_dq(tname, None, "uniqueness", "full_dup_percentage", full_dup_pct,
           100 - THRESHOLDS["uniqueness_pass"], status_full,
           "full_dup_pct_check", f"% duplicidade completa: {full_dup_pct:.4f}%")
    
    # Duplicidade por chave primária
    pk_cols_existing = [c for c in pk_cols if c in df.columns]
    if pk_cols_existing:
        key_dups = rc - df.select(*pk_cols_existing).distinct().count()
        key_dup_pct = (key_dups / rc * 100) if rc > 0 else 0
        
        # Verificar se a duplicidade é esperada (ex: installments_payments)
        if "legitimo" in pk_note.lower() or "repeticao" in pk_note.lower():
            status_key = "WARNING" if key_dups > 0 else "PASS"
            rule_desc = f"Duplicidade por chave (esperada — {pk_note}): {key_dups:,}"
        else:
            if key_dups == 0:
                status_key = "PASS"
            elif key_dup_pct < (100 - THRESHOLDS["uniqueness_pass"]):
                status_key = "PASS"
            elif key_dup_pct < (100 - THRESHOLDS["uniqueness_warning"]):
                status_key = "WARNING"
            else:
                status_key = "FAIL"
            rule_desc = f"Duplicidade por chave ({' + '.join(pk_cols_existing)}): {key_dups:,}"
        
        print(f"   Duplicidade por chave: {key_dups:,} ({key_dup_pct:.4f}%) → {status_key}")
        add_dq(tname, " + ".join(pk_cols_existing), "uniqueness", "key_duplicates", key_dups,
               0, status_key, "key_dup_check", rule_desc)
        add_dq(tname, " + ".join(pk_cols_existing), "uniqueness", "key_dup_percentage", key_dup_pct,
               100 - THRESHOLDS["uniqueness_pass"], status_key,
               "key_dup_pct_check", f"% duplicidade por chave: {key_dup_pct:.4f}%")
        
        # Score: 100 se 0 dups, senao 100 - dup_pct (com floor em 0)
        uniqueness_score = max(0, 100 - key_dup_pct)
    else:
        uniqueness_score = 100.0
    
    # Score final de uniqueness (média entre completa e por chave)
    uniqueness_score = max(0, 100 - max(full_dup_pct, 100 - uniqueness_score))
    if uniqueness_score >= THRESHOLDS["uniqueness_pass"]:
        status_u = "PASS"
    elif uniqueness_score >= THRESHOLDS["uniqueness_warning"]:
        status_u = "WARNING"
    else:
        status_u = "FAIL"
    
    TABLE_DIMENSION_SCORES.setdefault(tname, {})["uniqueness"] = uniqueness_score
    print(f"   → Score uniqueness: {uniqueness_score:.2f}% [{status_u}]")
    add_dq(tname, None, "uniqueness", "table_uniqueness_score", uniqueness_score,
           THRESHOLDS["uniqueness_pass"], status_u,
           "table_uniqueness_score", f"Score de unicidade: {uniqueness_score:.2f}%")

print(f"\n✅ Uniqueness avaliado para {len(SILVER_TABLES)} tabelas!")

In [0]:
# ============================================================================
# CÉLULA 5 — Validity (Tipos, Negativos, Domínios)
# ============================================================================
# Verifica tipos, valores negativos, chaves NULL e domínios categoricos.

sep = "─" * 70
print("=" * 70)
print("VALIDITY — TIPOS, NEGATIVOS, DOMÍNIOS")
print("=" * 70)

# Colunas que NÃO devem ter valores negativos
NON_NEGATIVE_COLS = {
    "application_train": ["CNT_CHILDREN", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"],
    "application_test": ["CNT_CHILDREN", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"],
    "bureau": ["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_MAX_OVERDUE"],
    "bureau_balance": [],
    "previous_application": ["AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE"],
    "pos_cash_balance": ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "SK_DPD", "SK_DPD_DEF"],
    "credit_card_balance": ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT"],
    "installments_payments": ["AMT_INSTALMENT", "AMT_PAYMENT"],
}

# Chaves que não devem ter NULL
KEY_COLUMNS = {
    "application_train": ["SK_ID_CURR"],
    "application_test": ["SK_ID_CURR"],
    "bureau": ["SK_ID_CURR", "SK_ID_BUREAU"],
    "bureau_balance": ["SK_ID_BUREAU"],
    "previous_application": ["SK_ID_PREV", "SK_ID_CURR"],
    "pos_cash_balance": ["SK_ID_PREV", "SK_ID_CURR"],
    "credit_card_balance": ["SK_ID_PREV", "SK_ID_CURR"],
    "installments_payments": ["SK_ID_PREV", "SK_ID_CURR"],
}

for tname in SILVER_TABLES:
    info = TABLE_INFO_CACHE.get(tname, {})
    if not info.get("exists"):
        continue
    
    df = info["df"]
    rc = info["row_count"]
    cols = info["columns"]
    col_names = [c[0] for c in cols]
    
    print(f"\n{'─' * 60}")
    print(f"📊 {tname}")
    print(f"{'─' * 60}")
    
    validity_issues = 0
    validity_checks = 0
    
    # 1. Chaves NULL
    for key_col in KEY_COLUMNS.get(tname, []):
        if key_col in col_names:
            key_nulls = df.filter(F.col(key_col).isNull()).count()
            validity_checks += 1
            if key_nulls == 0:
                status_v = "PASS"
            else:
                status_v = "FAIL"
                validity_issues += key_nulls
            print(f"   {key_col} NULL: {key_nulls} → {status_v}")
            add_dq(tname, key_col, "validity", "key_null_count", key_nulls, 0, status_v,
                   "key_null_check", f"Chave com NULL: {key_nulls}")
    
    # 2. Valores negativos em colunas que não deveriam ter
    for neg_col in NON_NEGATIVE_COLS.get(tname, []):
        if neg_col in col_names:
            neg_cnt = df.filter(F.col(neg_col) < 0).count()
            validity_checks += 1
            if neg_cnt == 0:
                status_v = "PASS"
            else:
                neg_pct = neg_cnt / rc * 100
                if neg_pct < 1:
                    status_v = "WARNING"
                else:
                    status_v = "FAIL"
                validity_issues += neg_cnt
                print(f"   {neg_col} negativos: {neg_cnt:,} ({neg_pct:.2f}%) → {status_v}")
            add_dq(tname, neg_col, "validity", "negative_count", neg_cnt, 0, status_v,
                   "negative_check", f"Valores negativos: {neg_cnt}")
    
    # 3. Verificar SK_ID_CURR = 0 (valor invalido)
    if "SK_ID_CURR" in col_names:
        zero_id = df.filter(F.col("SK_ID_CURR") == 0).count()
        validity_checks += 1
        if zero_id == 0:
            status_v = "PASS"
        else:
            status_v = "FAIL"
            validity_issues += zero_id
            print(f"   SK_ID_CURR = 0: {zero_id} → {status_v}")
        add_dq(tname, "SK_ID_CURR", "validity", "zero_id_count", zero_id, 0, status_v,
               "zero_id_check", f"SK_ID_CURR = 0: {zero_id}")
    
    # Score de validity
    if validity_checks > 0:
        validity_score = ((validity_checks - sum(1 for r in DQ_RESULTS 
            if r["table_name"] == tname and r["quality_dimension"] == "validity" 
            and r["status"] == "FAIL")) / validity_checks) * 100
    else:
        validity_score = 100.0
    
    TABLE_DIMENSION_SCORES.setdefault(tname, {})["validity"] = validity_score
    if validity_score >= THRESHOLDS["validity_pass"]:
        status_v = "PASS"
    elif validity_score >= THRESHOLDS["validity_warning"]:
        status_v = "WARNING"
    else:
        status_v = "FAIL"
    print(f"   → Score validity: {validity_score:.2f}% [{status_v}]")
    add_dq(tname, None, "validity", "table_validity_score", validity_score,
           THRESHOLDS["validity_pass"], status_v,
           "table_validity_score", f"Score de validade: {validity_score:.2f}%")

print(f"\n✅ Validity avaliado para {len(SILVER_TABLES)} tabelas!")

In [0]:
# ============================================================================
# CÉLULA 6 — Consistency (Consistência Interna)
# ============================================================================
# Verifica regras de consistência interna de cada tabela.
# Utiliza apenas colunas que realmente existem.

sep = "─" * 70
print("=" * 70)
print("CONSISTENCY — CONSISTÊNCIA INTERNA")
print("=" * 70)

# Regras de consistência por tabela
# Formato: (nome_regra, coluna_condicao, expressao_condicao, descricao)
CONSISTENCY_RULES = {
    "application_train": [
        ("amt_credit_positive", "AMT_CREDIT", F.col("AMT_CREDIT") > 0, "AMT_CREDIT deve ser positivo"),
        ("amt_income_positive", "AMT_INCOME_TOTAL", F.col("AMT_INCOME_TOTAL") > 0, "AMT_INCOME_TOTAL deve ser positivo"),
        ("children_nonneg", "CNT_CHILDREN", F.col("CNT_CHILDREN") >= 0, "CNT_CHILDREN deve ser >= 0"),
        ("target_binary", "TARGET", F.col("TARGET").isin([0, 1]), "TARGET deve ser 0 ou 1"),
    ],
    "application_test": [
        ("amt_credit_positive", "AMT_CREDIT", F.col("AMT_CREDIT") > 0, "AMT_CREDIT deve ser positivo"),
        ("amt_income_positive", "AMT_INCOME_TOTAL", F.col("AMT_INCOME_TOTAL") > 0, "AMT_INCOME_TOTAL deve ser positivo"),
        ("children_nonneg", "CNT_CHILDREN", F.col("CNT_CHILDREN") >= 0, "CNT_CHILDREN deve ser >= 0"),
    ],
    "bureau": [
        ("credit_active_valid", "CREDIT_ACTIVE", F.col("CREDIT_ACTIVE").isin(["Active", "Closed", "Bad debt", "Sold"]), "CREDIT_ACTIVE deve ser valido"),
        ("amt_credit_nonneg", "AMT_CREDIT_SUM", F.col("AMT_CREDIT_SUM") >= 0, "AMT_CREDIT_SUM deve ser >= 0"),
    ],
    "bureau_balance": [
        ("status_valid", "STATUS", F.col("STATUS").isin(["C", "0", "1", "2", "3", "4", "5", "X"]), "STATUS deve ser valido"),
        ("months_nonneg", "MONTHS_BALANCE", F.col("MONTHS_BALANCE") <= 0, "MONTHS_BALANCE deve ser <= 0"),
    ],
    "previous_application": [
        ("amt_application_nonneg", "AMT_APPLICATION", F.col("AMT_APPLICATION") >= 0, "AMT_APPLICATION deve ser >= 0"),
        ("amt_credit_nonneg", "AMT_CREDIT", F.col("AMT_CREDIT") >= 0, "AMT_CREDIT deve ser >= 0"),
    ],
    "pos_cash_balance": [
        ("contract_status_valid", "NAME_CONTRACT_STATUS", F.col("NAME_CONTRACT_STATUS").isin(["Active", "Completed", "Signed", "Approved", "Demand", "Returned", "Amortized debt", "Canceled", "XNA"]), "NAME_CONTRACT_STATUS deve ser valido"),
        ("months_nonpos", "MONTHS_BALANCE", F.col("MONTHS_BALANCE") <= 0, "MONTHS_BALANCE deve ser <= 0"),
    ],
    "credit_card_balance": [
        ("months_nonpos", "MONTHS_BALANCE", F.col("MONTHS_BALANCE") <= 0, "MONTHS_BALANCE deve ser <= 0"),
        ("credit_limit_nonneg", "AMT_CREDIT_LIMIT_ACTUAL", F.col("AMT_CREDIT_LIMIT_ACTUAL") >= 0, "AMT_CREDIT_LIMIT_ACTUAL deve ser >= 0"),
    ],
    "installments_payments": [
        ("amt_instalment_nonneg", "AMT_INSTALMENT", F.col("AMT_INSTALMENT") >= 0, "AMT_INSTALMENT deve ser >= 0"),
        ("amt_payment_nonneg", "AMT_PAYMENT", F.col("AMT_PAYMENT") >= 0, "AMT_PAYMENT deve ser >= 0"),
        ("days_instalment_neg", "DAYS_INSTALMENT", F.col("DAYS_INSTALMENT") < 0, "DAYS_INSTALMENT deve ser negativo"),
        ("instalment_number_pos", "NUM_INSTALMENT_NUMBER", F.col("NUM_INSTALMENT_NUMBER") >= 1, "NUM_INSTALMENT_NUMBER deve ser >= 1"),
    ],
}

for tname in SILVER_TABLES:
    info = TABLE_INFO_CACHE.get(tname, {})
    if not info.get("exists"):
        continue
    
    df = info["df"]
    rc = info["row_count"]
    col_names = [c[0] for c in info["columns"]]
    rules = CONSISTENCY_RULES.get(tname, [])
    
    print(f"\n{'─' * 60}")
    print(f"📊 {tname} ({len(rules)} regras)")
    print(f"{'─' * 60}")
    
    passed_rules = 0
    total_rules = 0
    
    for rule_name, col_name, condition, desc in rules:
        if col_name not in col_names:
            print(f"   ⏭️ {rule_name}: coluna {col_name} não existe — SKIP")
            continue
        
        total_rules += 1
        violations = df.filter(~condition | F.col(col_name).isNull()).count()
        violation_pct = (violations / rc * 100) if rc > 0 else 0
        
        if violations == 0:
            status_c = "PASS"
            passed_rules += 1
        elif violation_pct < 1:
            status_c = "WARNING"
        else:
            status_c = "FAIL"
        
        if violations > 0 or status_c != "PASS":
            print(f"   {rule_name}: {violations:,} violações ({violation_pct:.4f}%) → {status_c}")
        else:
            print(f"   {rule_name}: 0 violações → {status_c}")
        
        add_dq(tname, col_name, "consistency", f"rule_{rule_name}", violations, 0, status_c,
               rule_name, desc)
    
    # Score de consistency
    if total_rules > 0:
        consistency_score = (passed_rules / total_rules) * 100
    else:
        consistency_score = 100.0
    
    TABLE_DIMENSION_SCORES.setdefault(tname, {})["consistency"] = consistency_score
    if consistency_score >= THRESHOLDS["consistency_pass"]:
        status_c = "PASS"
    elif consistency_score >= THRESHOLDS["consistency_warning"]:
        status_c = "WARNING"
    else:
        status_c = "FAIL"
    print(f"   → Score consistency: {consistency_score:.2f}% [{status_c}]")
    add_dq(tname, None, "consistency", "table_consistency_score", consistency_score,
           THRESHOLDS["consistency_pass"], status_c,
           "table_consistency_score", f"Score de consistência: {consistency_score:.2f}%")

print(f"\n✅ Consistency avaliado para {len(SILVER_TABLES)} tabelas!")

In [0]:
# ============================================================================
# CÉLULA 7 — Referential Integrity (Integridade Referencial)
# ============================================================================
# Valida relacionamentos entre tabelas Silver.

sep = "─" * 70
print("=" * 70)
print("REFERENTIAL INTEGRITY — INTEGRIDADE REFERENCIAL")
print("=" * 70)

ref_scores_by_table = {}

for rel in REFERENTIAL_RELATIONSHIPS:
    src_table = rel["source"]
    tgt_table = rel["target"]
    src_key = rel["source_key"]
    tgt_key = rel["target_key"]
    
    src_info = TABLE_INFO_CACHE.get(src_table, {})
    tgt_info = TABLE_INFO_CACHE.get(tgt_table, {})
    
    if not src_info.get("exists") or not tgt_info.get("exists"):
        print(f"\n   ⚠️ {src_table}.{src_key} → {tgt_table}.{tgt_key}: tabela nao existe")
        add_dq(f"{src_table}→{tgt_table}", src_key, "referential_integrity", "table_exists", 0, 1, "WARNING",
               "ref_table_exists", "Tabela de origem ou destino nao existe")
        continue
    
    src_df = src_info["df"]
    tgt_df = tgt_info["df"]
    src_cols = [c[0] for c in src_info["columns"]]
    tgt_cols = [c[0] for c in tgt_info["columns"]]
    
    if src_key not in src_cols or tgt_key not in tgt_cols:
        print(f"\n   ⏭️ {src_table}.{src_key} → {tgt_table}.{tgt_key}: coluna nao existe")
        add_dq(f"{src_table}→{tgt_table}", src_key, "referential_integrity", "column_exists", 0, 1, "WARNING",
               "ref_column_exists", "Coluna de chave nao existe")
        continue
    
    # Calcular chaves distintas na origem
    src_keys = src_df.select(src_key).distinct()
    src_keys_count = src_keys.count()
    
    # Chaves no destino
    tgt_keys = tgt_df.select(tgt_key).distinct()
    
    # Matching
    matched = src_keys.join(tgt_keys, src_keys[src_key] == tgt_keys[tgt_key], "inner").count()
    unmatched = src_keys_count - matched
    integrity_pct = (matched / src_keys_count * 100) if src_keys_count > 0 else 100
    
    # Classificação
    if integrity_pct >= THRESHOLDS["ref_integrity_pass"]:
        status_r = "PASS"
    elif integrity_pct >= THRESHOLDS["ref_integrity_warning"]:
        status_r = "WARNING"
    else:
        status_r = "FAIL"
    
    print(f"\n   {src_table}.{src_key} → {tgt_table}.{tgt_key}")
    print(f"      Chaves na origem: {src_keys_count:,}")
    print(f"      Correspondidas: {matched:,} ({integrity_pct:.2f}%)")
    print(f"      Sem correspondência: {unmatched:,} ({100-integrity_pct:.2f}%) → {status_r}")
    
    add_dq(f"{src_table}→{tgt_table}", src_key, "referential_integrity", "matched_keys", matched,
           src_keys_count, status_r, "ref_match_check", f"Chaves correspondidas: {matched}/{src_keys_count}")
    add_dq(f"{src_table}→{tgt_table}", src_key, "referential_integrity", "unmatched_keys", unmatched,
           0, status_r, "ref_unmatch_check", f"Chaves sem correspondência: {unmatched}")
    add_dq(f"{src_table}→{tgt_table}", src_key, "referential_integrity", "integrity_percentage", integrity_pct,
           THRESHOLDS["ref_integrity_pass"], status_r,
           "ref_integrity_pct", f"Integridade: {integrity_pct:.2f}%")
    
    # Acumular score por tabela de origem
    ref_scores_by_table.setdefault(src_table, []).append(integrity_pct)

# Score de ref_integrity por tabela
for tname in SILVER_TABLES:
    scores = ref_scores_by_table.get(tname, [100.0])
    ref_score = sum(scores) / len(scores) if scores else 100
    TABLE_DIMENSION_SCORES.setdefault(tname, {})["referential_integrity"] = ref_score
    
    if ref_score >= THRESHOLDS["ref_integrity_pass"]:
        status_r = "PASS"
    elif ref_score >= THRESHOLDS["ref_integrity_warning"]:
        status_r = "WARNING"
    else:
        status_r = "FAIL"
    
    if tname in ref_scores_by_table:
        print(f"\n   → {tname} Score ref_integrity: {ref_score:.2f}% [{status_r}]")
        add_dq(tname, None, "referential_integrity", "table_ref_integrity_score", ref_score,
               THRESHOLDS["ref_integrity_pass"], status_r,
               "table_ref_integrity_score", f"Score de integridade referencial: {ref_score:.2f}%")

print(f"\n✅ Integridade referencial avaliada para {len(REFERENTIAL_RELATIONSHIPS)} relacionamentos!")

In [0]:
# ============================================================================
# CÉLULA 8 — TARGET Quality (Distribuição do Target)
# ============================================================================
# Avalia a distribuição da variável TARGET em application_train.

sep = "─" * 70
print("=" * 70)
print("TARGET QUALITY — DISTRIBUIÇÃO DO TARGET")
print("=" * 70)

info = TABLE_INFO_CACHE.get("application_train", {})
if info.get("exists") and "TARGET" in [c[0] for c in info["columns"]]:
    df = info["df"]
    rc = info["row_count"]
    
    # Distribuição
    target_dist = df.groupBy("TARGET").count().orderBy("TARGET").collect()
    
    print(f"\n📊 Distribuição do TARGET (application_train):")
    print(f"   Total de registros: {rc:,}")
    
    target_nulls = df.filter(F.col("TARGET").isNull()).count()
    if target_nulls > 0:
        print(f"   ⚠️ TARGET NULL: {target_nulls}")
        add_dq("application_train", "TARGET", "validity", "target_null", target_nulls, 0, "FAIL",
               "target_null_check", f"TARGET com NULL: {target_nulls}")
    else:
        add_dq("application_train", "TARGET", "validity", "target_null", 0, 0, "PASS",
               "target_null_check", "TARGET sem NULL")
    
    for row in target_dist:
        t = row["TARGET"]
        c = row["count"]
        pct = c / rc * 100
        print(f"   TARGET={t}: {c:>10,} ({pct:.2f}%)")
        add_dq("application_train", "TARGET", "consistency", f"target_{t}_count", c, 0, "PASS",
               f"target_{t}_dist", f"TARGET={t}: {c} ({pct:.2f}%)")
    
    # Verificar se é binario
    distinct_target = df.select("TARGET").distinct().count()
    if distinct_target == 2:
        print(f"\n   ✅ TARGET é binário (0 e 1)")
        add_dq("application_train", "TARGET", "validity", "target_binary", 1, 1, "PASS",
               "target_binary_check", "TARGET é binario")
    else:
        print(f"\n   ⚠️ TARGET tem {distinct_target} valores distintos")
        add_dq("application_train", "TARGET", "validity", "target_binary", 0, 1, "WARNING",
               "target_binary_check", f"TARGET tem {distinct_target} valores distintos")
    
    # Desbalanceamento
    if len(target_dist) == 2:
        t0 = target_dist[0]["count"]
        t1 = target_dist[1]["count"]
        ratio = max(t0, t1) / min(t0, t1)
        minority_pct = min(t0, t1) / rc * 100
        print(f"\n   📊 Desbalanceamento:")
        print(f"      Classe majoritária: {max(t0, t1):,} ({max(t0, t1)/rc*100:.2f}%)")
        print(f"      Classe minoritária: {min(t0, t1):,} ({minority_pct:.2f}%)")
        print(f"      Ratio: 1:{ratio:.2f}")
        
        if minority_pct < 10:
            status_t = "WARNING"
        else:
            status_t = "PASS"
        print(f"      Status: {status_t} (minoria < 10% → WARNING)")
        add_dq("application_train", "TARGET", "consistency", "class_imbalance_ratio", ratio, 10, status_t,
               "class_imbalance", f"Desbalanceamento: 1:{ratio:.2f} (minoría: {minority_pct:.2f}%)")
        add_dq("application_train", "TARGET", "consistency", "minority_class_pct", minority_pct, 10, status_t,
               "minority_pct", f"Classe minoritária: {minority_pct:.2f}%")
else:
    print("   ⚠️ Tabela application_train não existe ou coluna TARGET ausente")
    add_dq("application_train", "TARGET", "validity", "target_exists", 0, 1, "FAIL",
           "target_exists", "TARGET não encontrado")

print(f"\n✅ TARGET Quality avaliado!")

In [0]:
# ============================================================================
# CÉLULA 9 — Outliers (IQR em variáveis numéricas)
# ============================================================================
# Avalia outliers em variáveis numéricas principais usando IQR.
# NÃO remove outliers — apenas identifica e registra.

sep = "─" * 70
print("=" * 70)
print("OUTLIERS — MÉTODO IQR")
print("=" * 70)

# Variáveis numéricas principais para analisar outliers
OUTLIER_COLS = {
    "application_train": ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_CHILDREN"],
    "application_test": ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_CHILDREN"],
    "bureau": ["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_MAX_OVERDUE", "DAYS_CREDIT"],
    "bureau_balance": ["MONTHS_BALANCE"],
    "previous_application": ["AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT"],
    "pos_cash_balance": ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "SK_DPD", "SK_DPD_DEF"],
    "credit_card_balance": ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_CURRENT"],
    "installments_payments": ["AMT_INSTALMENT", "AMT_PAYMENT", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"],
}

for tname in SILVER_TABLES:
    info = TABLE_INFO_CACHE.get(tname, {})
    if not info.get("exists"):
        continue
    
    df = info["df"]
    rc = info["row_count"]
    col_names = [c[0] for c in info["columns"]]
    
    print(f"\n{'─' * 60}")
    print(f"📊 {tname}")
    print(f"{'─' * 60}")
    
    outlier_checks = 0
    outlier_warnings = 0
    
    for col_name in OUTLIER_COLS.get(tname, []):
        if col_name not in col_names:
            continue
        
        # Calcular Q1, Q3, IQR
        quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.01)
        if len(quantiles) < 2:
            continue
        
        q1, q3 = quantiles[0], quantiles[1]
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Contar outliers
        outlier_cnt = df.filter(
            (F.col(col_name) < lower_bound) | (F.col(col_name) > upper_bound)
        ).count()
        outlier_pct = (outlier_cnt / rc * 100) if rc > 0 else 0
        
        outlier_checks += 1
        
        # Classificar: outlier estatístico é WARNING, não FAIL
        if outlier_pct > 20:
            status_o = "WARNING"
            outlier_warnings += 1
        elif outlier_pct > 5:
            status_o = "WARNING"
        else:
            status_o = "PASS"
        
        if outlier_pct > 0.01:
            print(f"   {col_name:<35} Q1={q1:.2f} Q3={q3:.2f} IQR={iqr:.2f} outliers={outlier_cnt:,} ({outlier_pct:.2f}%) → {status_o}")
        
        add_dq(tname, col_name, "validity", "outlier_count_iqr", outlier_cnt, 0, status_o,
               "outlier_iqr", f"Outliers IQR: {outlier_cnt} ({outlier_pct:.2f}%) [Q1={q1:.2f}, Q3={q3:.2f}]")
    
    if outlier_checks == 0:
        print(f"   Nenhuma coluna numérica para analisar")

print(f"\n✅ Outliers avaliados!")
print(f"   ℹ️ Outliers são identificados mas NÃO removidos.")
print(f"   ℹ️ Outlier estatístico ≠ erro de dados (pode ser valor legítimo).")

In [0]:
# ============================================================================
# CÉLULA 10 — Quality Score (Cálculo do Score de Qualidade)
# ============================================================================
# Calcula o Quality Score de 0 a 100 para cada tabela, combinando 5 dimensoes.
# Pesos: completeness 30%, uniqueness 20%, validity 20%, consistency 15%, ref_integrity 15%

sep = "─" * 70
print("=" * 70)
print("QUALITY SCORE — CÁLCULO POR TABELA")
print("=" * 70)

TABLE_QUALITY_SCORES = {}

print(f"\n   Pesos das dimensoes:")
for dim, weight in DIMENSION_WEIGHTS.items():
    print(f"      {dim:<25} {weight*100:.0f}%")

for tname in SILVER_TABLES:
    scores = TABLE_DIMENSION_SCORES.get(tname, {})
    
    # Garantir que todas as dimensoes tenham score
    for dim in DIMENSION_WEIGHTS:
        if dim not in scores:
            scores[dim] = 100.0
    
    # Score ponderado
    quality_score = sum(scores.get(dim, 100) * weight for dim, weight in DIMENSION_WEIGHTS.items())
    TABLE_QUALITY_SCORES[tname] = quality_score
    
    # Classificação
    if quality_score >= THRESHOLDS["score_pass"]:
        status = "PASS"
    elif quality_score >= THRESHOLDS["score_warning"]:
        status = "WARNING"
    else:
        status = "FAIL"
    
    print(f"\n{'─' * 60}")
    print(f"📊 {tname}")
    print(f"{'─' * 60}")
    for dim in DIMENSION_WEIGHTS:
        s = scores.get(dim, 100)
        print(f"   {dim:<25} {s:>8.2f}%  (peso: {DIMENSION_WEIGHTS[dim]*100:.0f}%)")
    print(f"   {'─' * 40}")
    print(f"   SCORE FINAL: {quality_score:.2f}% [{status}]")
    
    add_dq(tname, None, "quality_score", "overall_score", quality_score,
           THRESHOLDS["score_pass"], status,
           "overall_quality_score", f"Score de qualidade: {quality_score:.2f}% (completeness {scores.get('completeness',0):.1f}%, uniqueness {scores.get('uniqueness',0):.1f}%, validity {scores.get('validity',0):.1f}%, consistency {scores.get('consistency',0):.1f}%, ref_integrity {scores.get('referential_integrity',0):.1f}%)")

print(f"\n{'=' * 70}")
print("RESUMO DOS SCORES")
print(f"{'=' * 70}")
for tname in SILVER_TABLES:
    score = TABLE_QUALITY_SCORES.get(tname, 0)
    if score >= THRESHOLDS["score_pass"]:
        status = "PASS"
    elif score >= THRESHOLDS["score_warning"]:
        status = "WARNING"
    else:
        status = "FAIL"
    print(f"   {tname:<35} {score:>8.2f}% [{status}]")

avg_score = sum(TABLE_QUALITY_SCORES.values()) / len(TABLE_QUALITY_SCORES) if TABLE_QUALITY_SCORES else 0
print(f"\n   Score médio geral: {avg_score:.2f}%")
print(f"\n✅ Quality Score calculado!")

In [0]:
# ============================================================================
# CÉLULA 11 — Data Quality Detail (Escrita da Tabela Detalhada)
# ============================================================================
# Grava todos os resultados DQ em credit_risk.silver.data_quality (APPEND).

sep = "─" * 70
print("=" * 70)
print("DATA QUALITY DETAIL — ESCRITA DA TABELA DETALHADA")
print("=" * 70)

print(f"\n   Total de resultados DQ acumulados: {len(DQ_RESULTS):,}")

# Schema da tabela data_quality
dq_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("quality_dimension", StringType(), True),
    StructField("metric_name", StringType(), True),
    StructField("metric_value", DoubleType(), True),
    StructField("threshold", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("rule_name", StringType(), True),
    StructField("rule_description", StringType(), True),
])

# Criar DataFrame dos resultados
dq_df = spark.createDataFrame(DQ_RESULTS, schema=dq_schema)

print(f"   Resultados a gravar: {dq_df.count()}")

# Gravar (APPEND para preservar histórico)
print(f"\n   Gravando em {DQ_DETAIL_TABLE}...")
dq_df.write.mode("append").format("delta").saveAsTable(DQ_DETAIL_TABLE)

print(f"   ✅ {len(DQ_RESULTS):,} registros gravados em {DQ_DETAIL_TABLE}")

# Estatisticas por dimensao
print(f"\n   Estatisticas por dimensao:")
dim_stats = spark.table(DQ_DETAIL_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).groupBy("quality_dimension", "status").count().orderBy("quality_dimension", "status").collect()
for r in dim_stats:
    print(f"      {r['quality_dimension']:<25} {r['status']:<10} {r['count']:>6}")

# Estatisticas por status
print(f"\n   Estatisticas por status:")
status_stats = spark.table(DQ_DETAIL_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).groupBy("status").count().orderBy(F.desc("count")).collect()
for r in status_stats:
    print(f"      {r['status']:<10} {r['count']:>6}")

print(f"\n✅ Data Quality Detail gravado!")

In [0]:
# ============================================================================
# CÉLULA 12 — Data Quality Summary (Resumo por Tabela)
# ============================================================================
# Cria credit_risk.silver.data_quality_summary com resumo por tabela.

sep = "─" * 70
print("=" * 70)
print("DATA QUALITY SUMMARY — RESUMO POR TABELA")
print("=" * 70)

summary_records = []

for tname in SILVER_TABLES:
    info = TABLE_INFO_CACHE.get(tname, {})
    if not info.get("exists"):
        summary_records.append({
            "execution_timestamp": EXECUTION_TIMESTAMP,
            "execution_id": EXECUTION_ID,
            "table_name": tname,
            "total_rules": 0,
            "passed_rules": 0,
            "warning_rules": 0,
            "failed_rules": 0,
            "quality_score": 0.0,
            "overall_status": "FAIL",
            "row_count": 0,
            "column_count": 0,
            "null_percentage": 100.0,
            "duplicate_count": 0,
            "referential_integrity_score": 0.0,
        })
        continue
    
    # Contar regras por status
    table_results = [r for r in DQ_RESULTS if r["table_name"] == tname or r["table_name"].startswith(tname + "→")]
    total_rules = len(table_results)
    passed = sum(1 for r in table_results if r["status"] == "PASS")
    warnings = sum(1 for r in table_results if r["status"] == "WARNING")
    failed = sum(1 for r in table_results if r["status"] == "FAIL")
    
    quality_score = TABLE_QUALITY_SCORES.get(tname, 0)
    
    if quality_score >= THRESHOLDS["score_pass"]:
        overall_status = "PASS"
    elif quality_score >= THRESHOLDS["score_warning"]:
        overall_status = "WARNING"
    else:
        overall_status = "FAIL"
    
    # NULL percentage
    rc = info["row_count"]
    cc = info["col_count"]
    cols = info["columns"]
    null_exprs = [F.sum(F.when(F.col(c[0]).isNull(), 1).otherwise(0)) for c in cols]
    null_result = info["df"].agg(*null_exprs).collect()[0]
    total_nulls = sum([null_result[i] for i in range(len(cols))])
    null_pct = (total_nulls / (rc * cc) * 100) if rc * cc > 0 else 0
    
    # Duplicates
    full_dups = rc - info["df"].dropDuplicates().count()
    
    # Ref integrity score
    ref_score = TABLE_DIMENSION_SCORES.get(tname, {}).get("referential_integrity", 100)
    
    summary_records.append({
        "execution_timestamp": EXECUTION_TIMESTAMP,
        "execution_id": EXECUTION_ID,
        "table_name": tname,
        "total_rules": total_rules,
        "passed_rules": passed,
        "warning_rules": warnings,
        "failed_rules": failed,
        "quality_score": float(quality_score),
        "overall_status": overall_status,
        "row_count": rc,
        "column_count": cc,
        "null_percentage": float(null_pct),
        "duplicate_count": full_dups,
        "referential_integrity_score": float(ref_score),
    })
    
    print(f"\n   {tname:<35} score={quality_score:>6.2f}% [{overall_status}] rules={total_rules} (P={passed} W={warnings} F={failed})")

# Schema
summary_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("total_rules", IntegerType(), True),
    StructField("passed_rules", IntegerType(), True),
    StructField("warning_rules", IntegerType(), True),
    StructField("failed_rules", IntegerType(), True),
    StructField("quality_score", DoubleType(), True),
    StructField("overall_status", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("column_count", IntegerType(), True),
    StructField("null_percentage", DoubleType(), True),
    StructField("duplicate_count", LongType(), True),
    StructField("referential_integrity_score", DoubleType(), True),
])

summary_df = spark.createDataFrame(summary_records, schema=summary_schema)

print(f"\n   Gravando em {DQ_SUMMARY_TABLE}...")
summary_df.write.mode("append").format("delta").saveAsTable(DQ_SUMMARY_TABLE)

print(f"   ✅ {len(summary_records)} registros gravados em {DQ_SUMMARY_TABLE}")

# Exibir resumo
display(spark.table(DQ_SUMMARY_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).orderBy(F.desc("quality_score")))

print(f"\n✅ Data Quality Summary gravado!")

In [0]:
# ============================================================================
# CÉLULA 13 — Integração com Auditoria
# ============================================================================
# Relaciona os resultados DQ com credit_risk.silver.audit_transformation.

sep = "─" * 70
print("=" * 70)
print("INTEGRAÇÃO COM AUDITORIA")
print("=" * 70)

try:
    audit_df = spark.table(AUDIT_TABLE)
    audit_count = audit_df.count()
    print(f"\n   Tabela de auditoria: {AUDIT_TABLE}")
    print(f"   Registros de auditoria: {audit_count}")
    
    # Exibir auditoria por tabela
    print(f"\n   Auditoria por tabela Silver:")
    audit_by_table = audit_df.select(
        "target_table", "source_row_count", "target_row_count",
        "records_removed", "execution_status", "processing_duration_seconds",
        "pipeline_version", "execution_timestamp"
    ).orderBy(F.col("execution_timestamp").desc()).collect()
    
    print(f"   {'Tabela':<45} {'Origem':>10} {'Destino':>10} {'Removidos':>10} {'Status':>10} {'Duração':>10}")
    print(f"   {sep}")
    for r in audit_by_table:
        tname = r['target_table'].replace('credit_risk.silver.', '')
        print(f"   {tname:<45} {r['source_row_count']:>10,} {r['target_row_count']:>10,} {r['records_removed']:>10} {r['execution_status']:>10} {r['processing_duration_seconds']:>10.1f}s")
    
    # Verificar consistencia: row_count Silver vs audit_transformation
    print(f"\n{'─' * 60}")
    print("CONSISTÊNCIA: Silver row_count vs Audit")
    print(f"{'─' * 60}")
    
    for tname in SILVER_TABLES:
        info = TABLE_INFO_CACHE.get(tname, {})
        if not info.get("exists"):
            continue
        
        silver_rc = info["row_count"]
        full_name = SILVER_TABLES_MAP[tname]
        
        # Buscar na auditoria
        audit_record = audit_df.filter(F.col("target_table") == full_name).orderBy(
            F.col("execution_timestamp").desc()).first()
        
        if audit_record:
            audit_rc = audit_record["target_row_count"]
            match = "✅" if silver_rc == audit_rc else "❌"
            print(f"   {match} {tname:<35} Silver={silver_rc:>10,}  Audit={audit_rc:>10,}")
            add_dq(tname, None, "consistency", "audit_row_count_match", 
                   1 if silver_rc == audit_rc else 0, 1,
                   "PASS" if silver_rc == audit_rc else "FAIL",
                   "audit_consistency", f"Silver row_count ({silver_rc}) vs Audit ({audit_rc})")
        else:
            print(f"   ⚠️ {tname}: sem registro de auditoria")
            add_dq(tname, None, "consistency", "audit_exists", 0, 1, "WARNING",
                   "audit_exists", "Sem registro na auditoria")
    
except Exception as e:
    print(f"   ❌ Erro ao acessar auditoria: {e}")
    add_dq("ALL", None, "consistency", "audit_access", 0, 1, "FAIL",
           "audit_access", f"Erro ao acessar auditoria: {e}")

print(f"\n✅ Integração com auditoria concluída!")

In [0]:
# ============================================================================
# CÉLULA 14 — Comparação Bronze → Silver
# ============================================================================
# Compara métricas entre Bronze e Silver para cada tabela.

sep = "─" * 70
print("=" * 70)
print("COMPARAÇÃO BRONZE → SILVER")
print("=" * 70)

print(f"\n   {'Tabela':<35} {'Bronze Rows':>12} {'Silver Rows':>12} {'Delta':>8} {'Bronze Cols':>12} {'Silver Cols':>12} {'Col Delta':>10}")
print(f"   {sep}")

for tname in SILVER_TABLES:
    silver_info = TABLE_INFO_CACHE.get(tname, {})
    if not silver_info.get("exists"):
        continue
    
    bronze_full = BRONZE_TABLES[tname]
    try:
        bronze_df = spark.table(bronze_full)
        bronze_rc = bronze_df.count()
        bronze_cc = len(bronze_df.columns)
    except:
        print(f"   ⚠️ {tname}: Bronze não existe")
        continue
    
    silver_rc = silver_info["row_count"]
    silver_cc = silver_info["col_count"]
    
    row_delta = silver_rc - bronze_rc
    col_delta = silver_cc - bronze_cc
    
    print(f"   {tname:<35} {bronze_rc:>12,} {silver_rc:>12,} {row_delta:>+8,} {bronze_cc:>12} {silver_cc:>12} {col_delta:>+10}")
    
    # Registrar métricas
    row_status = "PASS" if row_delta == 0 else ("WARNING" if abs(row_delta) < bronze_rc * 0.01 else "FAIL")
    add_dq(tname, None, "consistency", "bronze_vs_silver_rows", row_delta, 0, row_status,
           "row_count_comparison", f"Bronze={bronze_rc:,} Silver={silver_rc:,} Delta={row_delta:+,}")
    
    col_status = "PASS" if col_delta >= 0 else "WARNING"
    add_dq(tname, None, "consistency", "bronze_vs_silver_cols", col_delta, 0, col_status,
           "col_count_comparison", f"Bronze={bronze_cc} Silver={silver_cc} Delta={col_delta:+}")
    
    # Verificar que nenhum registro foi removido
    if row_delta == 0:
        print(f"      ✅ Nenhum registro removido")
    elif row_delta < 0:
        print(f"      ⚠️ {abs(row_delta):,} registros removidos")
    else:
        print(f"      ℹ️ {row_delta:,} registros adicionados")

# Resumo
print(f"\n{'─' * 60}")
print("RESUMO DA COMPARAÇÃO")
print(f"{'─' * 60}")
tables_no_removal = sum(1 for tname in SILVER_TABLES 
    if TABLE_INFO_CACHE.get(tname, {}).get("exists") and 
    TABLE_INFO_CACHE[tname]["row_count"] == spark.table(BRONZE_TABLES[tname]).count())
print(f"   Tabelas sem remoção de registros: {tables_no_removal}/{len(SILVER_TABLES)}")
print(f"   Todas as tabelas Silver têm mais colunas que Bronze (controle + flags)")

print(f"\n✅ Comparação Bronze → Silver concluída!")

In [0]:
# ============================================================================
# CÉLULA 15 — Principais Problemas (Dashboard-Ready)
# ============================================================================
# Identifica e ordena os principais problemas de qualidade.
# Estruturado para dashboards e monitoramento.

sep = "─" * 70
print("=" * 70)
print("PRINCIPAIS PROBLEMAS DE QUALIDADE")
print("=" * 70)

# Ler resultados da tabela DQ Detail (apenas desta execução)
dq_detail = spark.table(DQ_DETAIL_TABLE).filter(F.col("execution_id") == EXECUTION_ID)

# 1. Qualidade por tabela
print(f"\n{'─' * 60}")
print("QUALIDADE POR TABELA")
print(f"{'─' * 60}")

table_quality = spark.table(DQ_SUMMARY_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).select("table_name", "quality_score", "overall_status").orderBy(F.desc("quality_score"))

display(table_quality)

# 2. Qualidade por dimensao
print(f"\n{'─' * 60}")
print("QUALIDADE POR DIMENSÃO")
print(f"{'─' * 60}")

dim_quality = dq_detail.filter(
    F.col("quality_dimension").isin(list(DIMENSION_WEIGHTS.keys()))
).groupBy("table_name", "quality_dimension").agg(
    F.count("*").alias("total_checks"),
    F.sum(F.when(F.col("status") == "PASS", 1).otherwise(0)).alias("passed"),
    F.sum(F.when(F.col("status") == "WARNING", 1).otherwise(0)).alias("warnings"),
    F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0)).alias("failed"),
).orderBy("table_name", "quality_dimension")

display(dim_quality)

# 3. Principais problemas (FAIL e WARNING ordenados por impacto)
print(f"\n{'─' * 60}")
print("PRINCIPAIS PROBLEMAS (FAIL > WARNING)")
print(f"{'─' * 60}")

top_issues = dq_detail.filter(
    F.col("status").isin(["FAIL", "WARNING"])
).filter(
    F.col("quality_dimension") != "inspection"
).select(
    "table_name", "column_name", "quality_dimension",
    "metric_name", "metric_value", "status", "rule_description"
).orderBy(
    F.when(F.col("status") == "FAIL", 0).otherwise(1),
    F.col("metric_value").desc()
).limit(30)

display(top_issues)

# Contar problemas por status
fail_count = dq_detail.filter(F.col("status") == "FAIL").count()
warning_count = dq_detail.filter(F.col("status") == "WARNING").count()
pass_count = dq_detail.filter(F.col("status") == "PASS").count()

print(f"\n   Resumo de status:")
print(f"      PASS:     {pass_count:>6}")
print(f"      WARNING:  {warning_count:>6}")
print(f"      FAIL:     {fail_count:>6}")

# Problemas por tabela
print(f"\n{'─' * 60}")
print("PROBLEMAS POR TABELA")
print(f"{'─' * 60}")

issues_by_table = dq_detail.filter(
    F.col("status").isin(["FAIL", "WARNING"])
).groupBy("table_name", "status").count().orderBy("table_name", "status")

display(issues_by_table)

print(f"\n✅ Principais problemas identificados!")

In [0]:
# ============================================================================
# CÉLULA 16 — Validação Final
# ============================================================================
# Valida que as tabelas DQ foram criadas corretamente e exibe amostras.

sep = "─" * 70
print("=" * 70)
print("VALIDAÇÃO FINAL")
print("=" * 70)

# 1. Verificar tabela data_quality
dq_detail_count = spark.table(DQ_DETAIL_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).count()
print(f"\n   📊 {DQ_DETAIL_TABLE}")
print(f"      Registros desta execução: {dq_detail_count:,}")
print(f"      Total histórico: {spark.table(DQ_DETAIL_TABLE).count():,}")

# 2. Verificar tabela data_quality_summary
dq_summary_count = spark.table(DQ_SUMMARY_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).count()
print(f"\n   📊 {DQ_SUMMARY_TABLE}")
print(f"      Registros desta execução: {dq_summary_count}")
print(f"      Total histórico: {spark.table(DQ_SUMMARY_TABLE).count():,}")

# 3. Amostra da tabela data_quality_summary
print(f"\n{'─' * 60}")
print("AMOSTRA — data_quality_summary (esta execução)")
print(f"{'─' * 60}")
display(spark.table(DQ_SUMMARY_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).select("table_name", "quality_score", "overall_status",
         "total_rules", "passed_rules", "warning_rules", "failed_rules",
         "row_count", "column_count").orderBy(F.desc("quality_score")))

# 4. Amostra da tabela data_quality (top 20 FAIL/WARNING)
print(f"\n{'─' * 60}")
print("AMOSTRA — data_quality (top 20 FAIL/WARNING desta execução)")
print(f"{'─' * 60}")
display(spark.table(DQ_DETAIL_TABLE).filter(
    (F.col("execution_id") == EXECUTION_ID) &
    F.col("status").isin(["FAIL", "WARNING"])
).select("table_name", "column_name", "quality_dimension",
         "metric_name", "metric_value", "status", "rule_description"
).orderBy(
    F.when(F.col("status") == "FAIL", 0).otherwise(1),
    F.col("metric_value").desc()
).limit(20))

# 5. Verificar que nenhuma tabela Silver foi modificada
print(f"\n{'─' * 60}")
print("VERIFICAÇÃO: Nenhuma tabela Silver foi modificada")
print(f"{'─' * 60}")
print(f"   ✅ Este notebook NÃO modifica tabelas Silver (apenas LE e cria tabelas DQ)")
print(f"   ✅ Tabelas criadas/atualizadas: {DQ_DETAIL_TABLE}, {DQ_SUMMARY_TABLE}")

print(f"\n✅ Validação final concluída!")

In [0]:
# ============================================================================
# CÉLULA 17 — Resumo Executivo
# ============================================================================
# Apresenta o resumo final da qualidade da camada Silver.

sep = "=" * 50
print(sep)
print("SILVER DATA QUALITY - RESUMO")
print(sep)

# Contar tabelas por status
summary_df = spark.table(DQ_SUMMARY_TABLE).filter(
    F.col("execution_id") == EXECUTION_ID
).collect()

tables_pass = sum(1 for r in summary_df if r["overall_status"] == "PASS")
tables_warning = sum(1 for r in summary_df if r["overall_status"] == "WARNING")
tables_fail = sum(1 for r in summary_df if r["overall_status"] == "FAIL")
avg_score = sum(r["quality_score"] for r in summary_df) / len(summary_df) if summary_df else 0

print(f"\nTabelas analisadas: {len(summary_df)}")
print(f"\nPASS:    {tables_pass}")
print(f"WARNING: {tables_warning}")
print(f"FAIL:    {tables_fail}")
print(f"\nQuality Score médio: {avg_score:.2f}")

# Detalhes por tabela
print(f"\n{'─' * 50}")
print("DETALHES POR TABELA:")
print(f"{'─' * 50}")
for r in sorted(summary_df, key=lambda x: x["quality_score"], reverse=True):
    tname = r["table_name"]
    score = r["quality_score"]
    status = r["overall_status"]
    rc = r["row_count"]
    cc = r["column_count"]
    null_pct = r["null_percentage"]
    dups = r["duplicate_count"]
    ref_score = r["referential_integrity_score"]
    print(f"\n   {tname}:")
    print(f"      Score: {score:.2f}% [{status}]")
    print(f"      Registros: {rc:,} | Colunas: {cc}")
    print(f"      NULL%: {null_pct:.2f}% | Duplicidades: {dups}")
    print(f"      Integridade referencial: {ref_score:.2f}%")
    print(f"      Regras: {r['total_rules']} (P={r['passed_rules']} W={r['warning_rules']} F={r['failed_rules']})")

# Principais problemas
print(f"\n{'─' * 50}")
print("PRINCIPAIS PROBLEMAS:")
print(f"{'─' * 50}")

top_problems = spark.table(DQ_DETAIL_TABLE).filter(
    (F.col("execution_id") == EXECUTION_ID) &
    (F.col("status") == "WARNING") &
    F.col("quality_dimension").isin(list(DIMENSION_WEIGHTS.keys()))
).select("table_name", "quality_dimension", "metric_name", "rule_description").limit(10).collect()

if top_problems:
    for p in top_problems:
        print(f"   • {p['table_name']}: {p['rule_description']}")
else:
    print("   Nenhum problema significativo identificado.")

# Verificar FAILs
fail_problems = spark.table(DQ_DETAIL_TABLE).filter(
    (F.col("execution_id") == EXECUTION_ID) &
    (F.col("status") == "FAIL") &
    F.col("quality_dimension").isin(list(DIMENSION_WEIGHTS.keys()))
).select("table_name", "quality_dimension", "metric_name", "rule_description").limit(10).collect()

if fail_problems:
    print(f"\n   FAILS:")
    for p in fail_problems:
        print(f"   ❌ {p['table_name']}: {p['rule_description']}")

# Conclusão
print(f"\n{'─' * 50}")
print("CONCLUSÃO:")
print(f"{'─' * 50}")

silver_ready = tables_fail == 0 and avg_score >= THRESHOLDS["score_warning"]
if silver_ready:
    print(f"\n   Silver pronta para Gold: SIM")
    print(f"   ✅ Nenhuma tabela com FAIL")
    print(f"   ✅ Score médio >= {THRESHOLDS['score_warning']}")
else:
    print(f"\n   Silver pronta para Gold: NÃO")
    if tables_fail > 0:
        print(f"   ⚠️ {tables_fail} tabela(s) com FAIL")
    if avg_score < THRESHOLDS["score_warning"]:
        print(f"   ⚠️ Score médio ({avg_score:.2f}) < {THRESHOLDS['score_warning']}")
    print(f"\n   Recomendações:")
    print(f"   - Revisar tabelas com FAIL antes de avançar para Gold")
    print(f"   - Investigar WARNINGs que podem impactar ML")
    print(f"   - Documentar decisões sobre duplicidades legítimas")

print(f"\n{'─' * 50}")
print("EXECUÇÃO:")
print(f"{'─' * 50}")
print(f"   Execution ID: {EXECUTION_ID}")
print(f"   Batch ID: {BATCH_ID}")
print(f"   Pipeline: {PIPELINE_VERSION}")
print(f"   Resultados em: {DQ_DETAIL_TABLE}")
print(f"   Resumo em: {DQ_SUMMARY_TABLE}")

print(f"\n{sep}")
print("✅ SILVER DATA QUALITY CONCLUÍDO!")
print(sep)